In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ipywidgets import (
    VBox, HBox, IntSlider, FloatSlider, Button, Output, Layout
)
import random

# ---------- helpers ----------
def ngon(center, r, n, rotation):
    x0, y0 = center
    k = np.arange(n + 1)  # close polygon
    ang = rotation + 2 * np.pi * k / n
    x = x0 + r * np.cos(ang)
    y = y0 + r * np.sin(ang)
    return np.column_stack([x, y])

def build_segments(
    sides, poly_r, center_ring_r,
    groups, members, dtheta, rot_step, base_rot,
    intra_rad_shift, intra_rot_curve, intra_scale_step,
    group_phase, group_rot_step,
    center_jitter, rot_jitter,
    cx, cy, seed
):
    rng = np.random.default_rng(seed)
    segments = []
    group_span = (members - 1) * dtheta
    total_span = 2 * np.pi
    gap = total_span / groups - group_span

    for g in range(groups):
        theta_g = g * (group_span + gap) + group_phase + g * group_rot_step
        offsets = (np.arange(members) - (members - 1) / 2.0) * dtheta
        t_idx = (np.arange(members) - (members - 1) / 2.0) / max(1, (members - 1) / 2.0)

        for i, (dth, ti) in enumerate(zip(offsets, t_idx)):
            th = theta_g + dth
            rad = center_ring_r + intra_rad_shift * ti * poly_r
            px = cx + rad * np.cos(th)
            py = cy + rad * np.sin(th)

            if center_jitter > 0:
                px += (rng.random() - 0.5) * 2 * center_jitter * poly_r
                py += (rng.random() - 0.5) * 2 * center_jitter * poly_r

            # --- automatic radial alignment ---
            radial_angle = np.arctan2(py - cy, px - cx)

            # per member rotation: radial alignment + extras
            rot = radial_angle + base_rot + i * rot_step \
                  + intra_rot_curve * (ti**2) * np.sign(ti) \
                  + (rng.random()-0.5)*2*rot_jitter

            r_i = poly_r * (1 + intra_scale_step * ti)
            P = ngon((px, py), r_i, sides, rot)
            segments.append(P)

    return segments

def draw(
    sides=7, poly_r=0.18, center_ring_r=0.18,
    groups=10, members=7, dtheta=np.deg2rad(2.4),
    rot_step=np.deg2rad(4), base_rot=0.0,
    intra_rad_shift=0.00, intra_rot_curve=np.deg2rad(0.0), intra_scale_step=0.00,
    group_phase=0.0, group_rot_step=np.deg2rad(0.0),
    center_jitter=0.00, rot_jitter=np.deg2rad(0.0),
    color="teal", lw=0.6, alpha=0.95,
    figsize=(7,7), cx=0.0, cy=0.0, seed=0
):
    segments = build_segments(
        sides, poly_r, center_ring_r,
        groups, members, dtheta, rot_step, base_rot,
        intra_rad_shift, intra_rot_curve, intra_scale_step,
        group_phase, group_rot_step,
        center_jitter, rot_jitter,
        cx, cy, seed
    )

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect("equal")
    ax.axis("off")
    lim = center_ring_r + poly_r * (1.6 + abs(intra_scale_step))
    ax.set_xlim(cx - lim, cx + lim)
    ax.set_ylim(cy - lim, cy + lim)

    lc = LineCollection(segments, colors=color, linewidths=lw, alpha=alpha, joinstyle="miter")
    ax.add_collection(lc)
    return fig

# ---------- widgets ----------
w_sides  = IntSlider(value=7, min=3, max=24, step=1, description="sides")
w_poly_r = FloatSlider(value=0.18, min=0.02, max=0.4, step=0.002, description="poly_r")
w_ring_r = FloatSlider(value=0.18, min=0.02, max=0.6, step=0.002, description="center_r")

w_groups  = IntSlider(value=10, min=2, max=48, step=1, description="groups")
w_members = IntSlider(value=7, min=1, max=31, step=1, description="members")
w_dtheta  = FloatSlider(value=np.deg2rad(2.4), min=0.0, max=np.deg2rad(20), step=np.deg2rad(0.1), description="dtheta(rad)")
w_rotstep = FloatSlider(value=np.deg2rad(4), min=-np.deg2rad(30), max=np.deg2rad(30), step=np.deg2rad(0.2), description="rot_step")
w_brot    = FloatSlider(value=0.0, min=-np.pi, max=np.pi, step=np.deg2rad(1), description="base_rot")

w_intra_rad = FloatSlider(value=0.00, min=-0.5, max=0.5, step=0.01, description="intra_rad")
w_intra_rot = FloatSlider(value=np.deg2rad(0.0), min=-np.deg2rad(30), max=np.deg2rad(30), step=np.deg2rad(0.2), description="intra_rot")
w_intra_scale = FloatSlider(value=0.00, min=-0.8, max=0.8, step=0.01, description="intra_scale")

w_gphase = FloatSlider(value=0.0, min=0.0, max=2*np.pi, step=np.deg2rad(1), description="group_phase")
w_grot   = FloatSlider(value=np.deg2rad(0.0), min=-np.deg2rad(20), max=np.deg2rad(20), step=np.deg2rad(0.2), description="group_rot")

w_cjit = FloatSlider(value=0.00, min=0.0, max=0.2, step=0.005, description="center_jit")
w_rjit = FloatSlider(value=np.deg2rad(0.0), min=0.0, max=np.deg2rad(10), step=np.deg2rad(0.2), description="rot_jit")

w_lw   = FloatSlider(value=0.6, min=0.1, max=2.0, step=0.05, description="linewidth")
w_alpha= FloatSlider(value=0.95, min=0.05, max=1.0, step=0.01, description="alpha")
w_seed = IntSlider(value=0, min=0, max=10000, step=1, description="seed")

btn_render = Button(description="Render", button_style="primary", layout=Layout(width="120px"))
btn_save   = Button(description="Save SVG", layout=Layout(width="120px"))
out = Output()
_state = {"fig": None}

def current_kwargs():
    return dict(
        sides=w_sides.value, poly_r=w_poly_r.value, center_ring_r=w_ring_r.value,
        groups=w_groups.value, members=w_members.value, dtheta=w_dtheta.value,
        rot_step=w_rotstep.value, base_rot=w_brot.value,
        intra_rad_shift=w_intra_rad.value, intra_rot_curve=w_intra_rot.value, intra_scale_step=w_intra_scale.value,
        group_phase=w_gphase.value, group_rot_step=w_grot.value,
        center_jitter=w_cjit.value, rot_jitter=w_rjit.value,
        color="teal", lw=w_lw.value, alpha=w_alpha.value,
        figsize=(7,7), cx=0.0, cy=0.0, seed=w_seed.value
    )

def do_render(_=None):
    with out:
        out.clear_output(wait=True)
        fig = draw(**current_kwargs())
        _state["fig"] = fig
        plt.show(fig)

def do_save(_=None):
    if _state["fig"] is None:
        do_render()
    fname = f"grouped_ngons_{random.randint(10000,99999)}.svg"
    _state["fig"].savefig(fname, bbox_inches="tight", pad_inches=0.03)
    with out:
        print(f"saved: {fname}")

btn_render.on_click(do_render)
btn_save.on_click(do_save)

ui = VBox([
    HBox([w_sides, w_poly_r, w_ring_r]),
    HBox([w_groups, w_members, w_dtheta]),
    HBox([w_rotstep, w_brot, w_intra_rot]),
    HBox([w_intra_rad, w_intra_scale, w_gphase]),
    HBox([w_grot, w_cjit, w_rjit]),
    HBox([w_lw, w_alpha, w_seed, btn_render, btn_save]),
    out
])
display(ui)

# initial draw
do_render()
